In [ ]:
!pip install transformers datasets accelerate

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/Data set/ai-medical-chatbot[1].csv")

In [ ]:
df.head(6)

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...
5,Q. I had a surgery which ended up with some fa...,"Hello doctor,I had an emergency surgery six mo...",Hello. If you are saying it is already six mon...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256916 entries, 0 to 256915
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Description  256916 non-null  object
 1   Patient      256916 non-null  object
 2   Doctor       256916 non-null  object
dtypes: object(3)
memory usage: 5.9+ MB


In [ ]:
df.describe()

,Description,Patient,Doctor
count,256916,256916,256916
unique,228722,246006,242150
top,Q. Why do periods get delayed after first time...,"Hello doctor, My fiancee and I had unprotected...",Hi. For further doubts consult a sexologist on...
freq,1137,1137,1519


In [ ]:
df.isnull().sum()

,0
Description,0
Patient,0
Doctor,0


In [ ]:
df.duplicated().sum()

np.int64(10378)

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

In [ ]:
#Prepare Conversation Format (Patient ↔ Doctor)

In [ ]:
def prepared_conversation_between_doctor_and_patient(row):
    patient_question = "## Patient: " + row["Description"].strip()
    doctor_answer = "## Doctor: " + row["Doctor"].strip()
    conversation = {
        "patient_question" : patient_question,
        "doctor_answer" : doctor_answer
    }
    return conversation

In [ ]:
conversations = df.apply(prepared_conversation_between_doctor_and_patient, axis=1)
print("Number of Conversations between Doctor and Patients: ", len(conversations))

Number of Conversations between Doctor and Patients:  246538


In [ ]:
conversations.iloc[0]

{'patient_question': '## Patient: Q. What does abutment of the nerve root mean?',
 'doctor_answer': '## Doctor: Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}

In [ ]:
conversations.tolist()[:20]

[{'patient_question': '## Patient: Q. What does abutment of the nerve root mean?',
  'doctor_answer': '## Doctor: Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'},
 {'patient_question': '## Patient: Q. What should I do to reduce my weight gained due to genetic hypothyroidism?',
  'doctor_answer': '## Doctor: Hi. You have really done well with the hypothyroidism problem. Your levels are normal with less medications which are very good. As it is genetically induced, it is very difficult to lose weight. My advice to you is, you should focus on maintaining normal levels of TSH (thyroid-stimulating hormone) and try to remain active, having a positive outlook in life. Or else, it will become very difficult to balance your life with the symptoms of hypothyroidism. Even though your weight has not reduced, be very careful in not putting on weight here afterward. Everyday brisk walk

In [ ]:
#Convert Conversations into Hugging Face Dataset

In [ ]:
# convert prepared input (patient questions & doctor ansers ) into datadfame.
from datasets import Dataset

dataset = Dataset.from_list(conversations.tolist())

print("Number of training examples:", len(dataset))
print(dataset[0])

Number of training examples: 246538
{'patient_question': '## Patient: Q. What does abutment of the nerve root mean?', 'doctor_answer': '## Doctor: Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import transformers
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Training dataset (80%)
fraction = 0.8
train_size = int(fraction * len(dataset))

train_dataset = dataset.shuffle(seed=42).select(range(train_size))

# Validation dataset (20%)
eval_dataset = dataset.shuffle(seed=42).select(range(train_size, len(dataset)))

In [ ]:
print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

Train size: 197230
Eval size: 49308


In [ ]:
base_model_id = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id)



tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(conversations):
    texts = [
        patient_question + "\n" + doctor_answer
        for patient_question, doctor_answer in zip(conversations['patient_question'], conversations['doctor_answer'])
    ]
    tokenized_texts = tokenizer(texts, truncation=True, padding="max_length", max_length= 512 )
    tokenized_texts["labels"] = tokenized_texts["input_ids"].copy()
    return tokenized_texts

In [ ]:
# Tokenize the dataset
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

NameError: name 'train_dataset' is not defined

In [ ]:
output_dir = "./fine_tuned_medical_chatbot"

training_args = TrainingArguments(
    output_dir=output_dir,
    max_steps=300,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    disable_tqdm=False,
    report_to="none",
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    # eval_dataset = tokenized_eval

)

In [ ]:
## enable checkpointing + CUDA memory fix
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
model.gradient_checkpointing_enable()

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete.")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Starting training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,1.923000
2,1.819000
3,1.319900
4,1.001600
5,1.447900
6,1.984200
7,3.030500
8,0.805500
9,1.767100
10,1.161200


Step,Training Loss
1,1.923000
2,1.819000
3,1.319900
4,1.001600
5,1.447900
6,1.984200
7,3.030500
8,0.805500
9,1.767100
10,1.161200


Training complete.


In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Model and tokenizer saved Successfully.")

In [ ]:
#Reload Fine-Tuned Medical Chatbot Model for Inference
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_dir = "/content/drive/MyDrive/Data set/ai-medical-chatbot[1].csv"

# Load model & tokenizer
model = AutoModelForCausalLM.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Create a chatbot pipeline
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0
)

In [ ]:
import torch

def safe_generate(prompt, threshold=0.25):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=200,
            output_scores=True,
            return_dict_in_generate=True,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    scores = outputs.scores  # logits
    probs = [torch.softmax(s, dim=-1).max().item() for s in scores]
    avg_confidence = sum(probs) / len(probs)

    if avg_confidence < threshold:
        return "SORRY I DON'T KNOW"

    text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    return text.split("Doctor:")[-1].strip()


In [ ]:
prompt = "Patient: What are the symptoms of Lymphocytic Choriomeningitis (LCM)?\nDoctor:"
response = chatbot(prompt, max_length=200, do_sample=True, top_p=0.9, temperature=0.8)

print("Chatbot:", response[0]["generated_text"])